# 02 — MoE Expert Load Balancing, from scratch

Companion notebook to `../01-deepseek-architecture-deep-dive.md`.

This notebook simulates the Mixture-of-Experts load-balancing problem concretely: route a batch of
synthetic tokens through a router across several experts, first with **no load balancing at all**
(showing some experts overloaded and others nearly idle), then with a simple **bias-adjustment**
mechanism conceptually similar to DeepSeek's published auxiliary-loss-free load balancing — nudging
routing scores directly toward more even utilization, without adding any term to a training loss.

Fully offline: numpy only, no model downloads, no GPU, no API keys.

In [1]:
import numpy as np

np.random.seed(7)

num_experts = 8
num_tokens = 4000
hidden_dim = 32

# Random synthetic token representations and a router weight matrix
tokens = np.random.randn(num_tokens, hidden_dim)
W_router = np.random.randn(hidden_dim, num_experts)

# Deliberately bias a few experts to be "attractive" to the router -- this mimics what happens
# organically during training absent any balancing mechanism: the router converges on favoring
# a small subset of experts, starving the rest of gradient signal.
W_router[:, :3] *= 2.5

print(f"num_experts={num_experts}, num_tokens={num_tokens}, hidden_dim={hidden_dim}")
print("Router weight matrix shape:", W_router.shape)


num_experts=8, num_tokens=4000, hidden_dim=32
Router weight matrix shape: (32, 8)


## 1. Routing with NO load balancing

Each token is routed to its single highest-scoring expert (top-1 routing). With no correction
mechanism, the router's bias toward experts 0-2 (set above) directly translates into wildly uneven
utilization.

In [2]:
def route(tokens: np.ndarray, W_router: np.ndarray, bias: np.ndarray = None) -> np.ndarray:
    """Top-1 routing: each token goes to the single expert with the highest score."""
    logits = tokens @ W_router
    if bias is not None:
        logits = logits + bias
    return np.argmax(logits, axis=1)


def utilization(assignments: np.ndarray, num_experts: int) -> np.ndarray:
    return np.bincount(assignments, minlength=num_experts)


def print_utilization(counts: np.ndarray, num_tokens: int, label: str):
    print(label)
    max_bar = 60
    for e, c in enumerate(counts):
        bar_len = int(c / num_tokens * max_bar * num_experts / 2) if num_tokens else 0
        bar = "#" * min(bar_len, max_bar)
        print(f"  expert {e}: {c:5d} tokens  {bar}")


assignments_no_balance = route(tokens, W_router)
counts_no_balance = utilization(assignments_no_balance, num_experts)
print_utilization(counts_no_balance, num_tokens, "Utilization with NO load balancing:")

imbalance_ratio = counts_no_balance.max() / max(counts_no_balance.min(), 1)
print(f"\nMax/min utilization ratio (no balancing): {imbalance_ratio:.1f}x")
print(f"Ideal, perfectly even utilization would be {num_tokens / num_experts:.0f} tokens per expert.")


Utilization with NO load balancing:
  expert 0:   686 tokens  #########################################
  expert 1:  1109 tokens  ############################################################
  expert 2:   909 tokens  ######################################################
  expert 3:   279 tokens  ################
  expert 4:   322 tokens  ###################
  expert 5:   210 tokens  ############
  expert 6:   229 tokens  #############
  expert 7:   256 tokens  ###############

Max/min utilization ratio (no balancing): 5.3x
Ideal, perfectly even utilization would be 500 tokens per expert.


## 2. Auxiliary-loss-free-style bias balancing

Instead of adding a load-balancing term to a training loss (which would pull routing decisions away
from quality-driven choices to force balance), DeepSeek's published approach maintains a per-expert
**bias** added directly to routing scores: nudge an underused expert's bias up (making it more likely
to be picked next round) and an overused expert's bias down — a direct, score-level control loop,
decoupled from the model's actual training objective.

We simulate that same directional idea here: run several rounds of routing, and after each round,
adjust `bias` based on how far each expert's utilization was from the even-utilization target.

In [3]:
bias = np.zeros(num_experts)
target = num_tokens / num_experts
lr = 2.0  # bias adjustment rate

n_rounds = 30
history = []
for rnd in range(n_rounds):
    assignments = route(tokens, W_router, bias)
    counts = utilization(assignments, num_experts)
    history.append(counts.copy())
    deviation = counts - target                 # positive = overused, negative = underused
    bias -= lr * (deviation / num_tokens)        # push bias DOWN for overused, UP for underused

final_assignments = route(tokens, W_router, bias)
final_counts = utilization(final_assignments, num_experts)
print_utilization(final_counts, num_tokens, "Utilization AFTER auxiliary-loss-free-style bias balancing:")

final_ratio = final_counts.max() / max(final_counts.min(), 1)
print(f"\nMax/min utilization ratio (after balancing): {final_ratio:.2f}x  "
      f"(was {imbalance_ratio:.1f}x with no balancing)")

assert final_ratio < imbalance_ratio, "balancing should reduce the utilization imbalance"
print("\nOK: the bias-based adjustment materially reduced the max/min utilization imbalance, "
      "and it never added anything to a training loss -- it only ever nudges routing SCORES "
      "directly, round over round, the same directional idea as DeepSeek's published "
      "auxiliary-loss-free load balancing.")


Utilization AFTER auxiliary-loss-free-style bias balancing:
  expert 0:   539 tokens  ################################
  expert 1:   644 tokens  ######################################
  expert 2:   603 tokens  ####################################
  expert 3:   460 tokens  ###########################
  expert 4:   446 tokens  ##########################
  expert 5:   442 tokens  ##########################
  expert 6:   439 tokens  ##########################
  expert 7:   427 tokens  #########################

Max/min utilization ratio (after balancing): 1.51x  (was 5.3x with no balancing)

OK: the bias-based adjustment materially reduced the max/min utilization imbalance, and it never added anything to a training loss -- it only ever nudges routing SCORES directly, round over round, the same directional idea as DeepSeek's published auxiliary-loss-free load balancing.


## 3. Watching the imbalance shrink round by round

In [4]:
print(f"{'round':>6} | {'max/min ratio':>14} | {'std of utilization':>20}")
print("-" * 48)
for i, counts in enumerate(history):
    ratio = counts.max() / max(counts.min(), 1)
    std = counts.std()
    if i % 5 == 0 or i == n_rounds - 1:
        print(f"{i:>6} | {ratio:>14.2f} | {std:>20.1f}")


 round |  max/min ratio |   std of utilization
------------------------------------------------
     0 |           5.28 |                329.8
     5 |           3.66 |                264.1
    10 |           2.73 |                206.8
    15 |           2.22 |                164.9
    20 |           1.86 |                126.0
    25 |           1.67 |                100.9
    29 |           1.54 |                 82.7


## 4. Tying it back

- **No balancing** (Section 1) reproduces the real training problem plainly: a router with any
  organic preference for a subset of experts converges on using them heavily while starving the
  rest, visible directly in the lopsided utilization bars.
- The **bias adjustment** (Section 2) fixes this without ever touching a loss function — it only
  ever adjusts routing *scores* directly, based on recent utilization, exactly the mechanism
  DeepSeek-V3's published auxiliary-loss-free load balancing uses instead of a traditional auxiliary
  loss term that would otherwise trade away some model quality for the sake of even utilization.
- The round-by-round view (Section 3) shows the imbalance shrinking progressively rather than
  jumping to perfectly even instantly — a realistic picture of a control loop converging, not a
  one-shot fix.